In [15]:
import polars as pl
import torch.nn as nn
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
import torch
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
from torch_geometric.nn import GCNConv, global_mean_pool
import torch.nn.functional as F
from sklearn.metrics import r2_score
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import mlflow
from rdkit.Chem import rdFingerprintGenerator
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

pl.Config.set_tbl_cols(-1)       # Wyświetl wszystkie kolumny
pl.Config.set_tbl_rows(20)       # Ustaw limit wierszy (lub -1 dla wszystkich)
pl.Config.set_fmt_str_lengths(100) # Nie skracaj długich ciągów (np. SMILES)

polars.config.Config

In [16]:
df = pl.read_parquet("processed_data/ChEMBL_processed.parquet")

In [17]:
print(df.columns)

['activity_id', 'molregno', 'canonical_smiles', 'mw_freebase', 'alogp', 'hba', 'hbd', 'psa', 'rtb', 'aromatic_rings', 'qed_weighted', 'standard_value', 'standard_units', 'standard_type', 'standard_relation', 'pchembl_value', 'target_chembl_id', 'target_name', 'confidence_score', 'pIC50']


In [18]:
radius = 2
n_bits = 2048
mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)

def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return MurckoScaffold.GetScaffoldForMol(mol)

# Konwersja SMILES -> Fingerprint
def smiles_to_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros((n_bits,), dtype=np.float32)
    # GetFingerprintAsNumPy zwraca od razu tablicę numpy
    return mfpgen.GetFingerprintAsNumPy(mol).astype(np.float32)

# Architektura zgodna z wytycznymi
class MLPBaseline(nn.Module):
    def __init__(self, input_size=2048):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.network(x)

class GNNBaseline(torch.nn.Module):
    def __init__(self, node_features):
        super(GNNBaseline, self).__init__()
        # Warstwy grafowe
        self.conv1 = GCNConv(node_features, 64)
        self.conv2 = GCNConv(64, 64)
        self.conv3 = GCNConv(64, 64)
        
        # Head liniowy (regresja)
        self.lin1 = torch.nn.Linear(64, 32)
        self.lin2 = torch.nn.Linear(32, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        
        # 1. Message Passing
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        
        # 2. Pooling (cała molekuła)
        x = global_mean_pool(x, batch)
        
        # 3. Regresja
        x = F.relu(self.lin1(x))
        return self.lin2(x)

In [19]:
## Uczenie
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLPBaseline().to(device) # lub GNNBaseline
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.MSELoss()

def train_step(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for batch in loader:
        # Dla MLP: x = batch[0].to(device), y = batch[1].to(device)
        # Dla GNN: batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch) 
        loss = criterion(out, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [20]:
def train_one_epoch(model, loader, optimizer, criterion, device, is_gnn=False):
    model.train()
    total_loss = 0
    for batch in loader:
        if is_gnn:
            batch = batch.to(device)
            target = batch.y.view(-1, 1)
            out = model(batch)
        else:
            x, y = batch
            x, target = x.to(device), y.to(device).view(-1, 1)
            out = model(x)

        loss = criterion(out, target)
        
        # Sprawdzenie czy loss nie jest NaN
        if torch.isnan(loss):
            print("Loss exploded to NaN! Stopping...")
            return float('nan')

        optimizer.zero_grad()
        loss.backward()
        
        # --- DODAJ TO: Gradient Clipping ---
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, device, is_gnn=False):
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch in loader:
            if is_gnn:
                batch = batch.to(device)
                out = model(batch)
                target = batch.y
            else:
                x, y = batch
                out = model(x.to(device))
                target = y
            
            all_preds.append(out.cpu().numpy())
            all_targets.append(target.numpy())
            
    all_preds = np.concatenate(all_preds).ravel()
    all_targets = np.concatenate(all_targets).ravel()
    
    # --- DIAGNOSTYKA ---
    if np.isnan(all_preds).any():
        print(f"BŁĄD: Predykcje modelu zawierają NaN! (Pierwsze 5: {all_preds[:5]})")
    if np.isnan(all_targets).any():
        print("BŁĄD: Etykiety w zbiorze walidacyjnym zawierają NaN!")
    
    return r2_score(all_targets, all_preds)

In [21]:
def get_valid_fp(smiles):
    """Próbuje stworzyć molekułę i fingerprint. Zwraca None przy błędzie."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        # Próba wygenerowania fingerprinta
        return mfpgen.GetFingerprintAsNumPy(mol).astype(np.float32)
    except:
        return None

# 1. Dodajemy kolumnę z fingerprintami (jako obiekty)
df_temp = df.with_columns(
    pl.col("canonical_smiles").map_elements(get_valid_fp, return_dtype=pl.Object).alias("fp")
)

# 2. Usuwamy wiersze, których RDKit nie zrozumiał
df_clean = df_temp.filter(
    (pl.col("fp").is_not_null()) & 
    (pl.col("pIC50").is_not_null()) &
    (pl.col("pIC50").is_not_nan())
)
print(f"Liczba próbek po pełnym oczyszczeniu: {len(df_clean)}")

# 3. Informacja dla Ciebie, ile danych wypadło
print(f"Odrzucono {len(df) - len(df_clean)} błędnych cząsteczek.")

# 4. Przygotowanie X i y z oczyszczonego zbioru
X = np.stack(df_clean["fp"].to_list())
y = df_clean["pIC50"].to_numpy().astype(np.float32)

# 5. Teraz możesz bezpiecznie robić train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Definicja loaderów (jak wcześniej)
train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val)), batch_size=64)

is_gnn_flag = False

[11:09:53] WARNING: not removing hydrogen atom without neighbors
[11:10:07] Can't kekulize mol.  Unkekulized atoms: 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77
[11:10:27] Explicit valence for atom # 1 P, 7, is greater than permitted
[11:10:55] Explicit valence for atom # 34 P, 7, is greater than permitted
[11:11:16] WARNING: not removing hydrogen atom without neighbors


Liczba próbek po pełnym oczyszczeniu: 561058
Odrzucono 4 błędnych cząsteczek.


In [ ]:
## Główna petla treningowa z MLflow

# Konfiguracja
EPOCHS = 100
LR = 0.0001
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Inicjalizacja modelu, optimizera i straty
# model = MLPBaseline().to(DEVICE) # Odkomentuj właściwy
# is_gnn_flag = False

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = torch.nn.MSELoss()

# Start eksperymentu MLflow
with mlflow.start_run(run_name="Baseline_Model_Experiment"):
    # Logowanie hiperparametrów
    mlflow.log_param("lr", LR)
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("model_type", "MLP") # lub GNN
    
    for epoch in range(EPOCHS):
        # Trening
        avg_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, is_gnn=is_gnn_flag)
        
        # Ewaluacja na zbiorze walidacyjnym
        r2_val = evaluate(model, val_loader, DEVICE, is_gnn=is_gnn_flag)
        
        # Logowanie metryk do MLflow
        mlflow.log_metric("mse_loss", avg_loss, step=epoch)
        mlflow.log_metric("r2_val", r2_val, step=epoch)
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch}: Loss {avg_loss:.4f} | R2 Val: {r2_val:.4f}")

    # Logowanie końcowego modelu
    mlflow.pytorch.log_model(model, "model")

Epoch 0: Loss 1.8311 | R2 Val: 0.4361
Epoch 10: Loss 0.6639 | R2 Val: 0.5922
Epoch 20: Loss 0.4816 | R2 Val: 0.5925
Epoch 30: Loss 0.3906 | R2 Val: 0.5899
Epoch 40: Loss 0.3357 | R2 Val: 0.5847
Epoch 50: Loss 0.3013 | R2 Val: 0.5838
Epoch 60: Loss 0.2769 | R2 Val: 0.5807
Epoch 70: Loss 0.2606 | R2 Val: 0.5794
Epoch 80: Loss 0.2468 | R2 Val: 0.5801


In [ ]:
print(np.isnan(X).any())

False
